# Day 13: Unsupervised Learning, Customer Segmentation with K-Means

Unsupervised learning trains without an answer column. The algorithm looks at raw rows and finds groups by itself. The main families are:

- Clustering: grouping similar points (K-Means, hierarchical clustering, DBSCAN)
- Association: finding rules between variables (Apriori, FP-Growth), for example market basket analysis
- Dimensionality reduction: compressing data while keeping its structure (PCA, t-SNE)

The project here segments customers of a mall into groups by income and spending so that marketing can be aimed at each slice differently.

## Project Setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

## Loading the Dataset

The Mall Customers dataset comes from Kaggle through kagglehub, which drops files into a local cache without needing an API key.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("joebeachcapital/customer-segmentation")
print("Dataset folder:", path)

csv_file = None
for root, dirs, files in os.walk(path):
    for file_name in files:
        if file_name.endswith(".csv"):
            csv_file = os.path.join(root, file_name)

print("CSV found at:", csv_file)
df = pd.read_csv(csv_file)
print("Dataset loaded successfully.")

## Initial Data Analysis

Before clustering, the structure of the data has to be understood: shape, columns, types, missing values and duplicates.

### Shape and Columns

In [ ]:
print(df.shape)
df.columns

### Information and Types

In [ ]:
df.info()

Types to know: int64 counts whole numbers, float64 decimals, object text or categories. The info output doubles as a missing value report because the non-null counts are shown per column.

### Statistical Summary

In [ ]:
df.describe()

The mean, standard deviation, minimum, maximum and quartiles per column reveal the ranges and flag anything that looks unusual.

### Missing Values

In [ ]:
df.isnull().sum()

### Duplicates

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df[df.duplicated()]

### Numerical and Categorical Columns

In [ ]:
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = df.select_dtypes(include=["object"]).columns

print("Numerical :", list(numerical_columns))
print("Categorical:", list(categorical_columns))

Numerical features carry measurable amounts such as income, categorical features carry labels such as gender.

### Unique Values in Categorical Columns

In [ ]:
for col in categorical_columns:
    print(f"\nUnique values in {col}: {df[col].unique()}")

Checking the categories exposes spelling problems or inconsistent labels before encoding happens.

## Data Cleaning

Clustering is sensitive to noise, so duplicates go and missing values get filled.

### Removing Duplicates

In [ ]:
df.drop_duplicates(inplace=True)
print("Duplicates now:", df.duplicated().sum())
print("Rows remaining:", len(df))

### Handling Missing Values

Numeric columns take the median and categorical columns take the mode. The Mall Customers data has none in this run, but the handling code stays in the pipeline for robustness.

In [ ]:
for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values now:", df.isnull().sum().sum())

### Outlier Detection

K-Means centroids are averages, so a few extreme rows can drag a whole cluster toward them. Boxplots show the whisker range and IQR counting puts numbers on the problem.

In [ ]:
plt.figure(figsize=(15, 8))
for idx, col in enumerate(numerical_columns):
    plt.subplot(2, 3, idx + 1)
    sns.boxplot(y=df[col])
    plt.title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()

In [ ]:
for col in numerical_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR_value = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR_value
    upper_limit = Q3 + 1.5 * IQR_value
    outlier_rows = df[(df[col] < lower_limit) | (df[col] > upper_limit)]
    print(f"{col} : {len(outlier_rows)} outliers")

IQR counts any row below Q1 - 1.5 times the spread or above Q3 + 1.5 times the spread. The spending scores show a handful of small outliers at the low end.

### Encoding Categorical Features

Algorithms cannot digest text directly, so Male and Female become 0 and 1.

In [ ]:
label_encoder = LabelEncoder()
for col in categorical_columns:
    df[col] = label_encoder.fit_transform(df[col])

df.head()

## Exploratory Data Analysis

### Univariate Analysis: Histograms

In [ ]:
df.hist(figsize=(15, 10), bins=18)
plt.tight_layout()
plt.show()

### Distribution Plots with Density Curves

In [ ]:
plt.figure(figsize=(15, 10))
for idx, col in enumerate(numerical_columns):
    plt.subplot(2, 3, idx + 1)
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

### Countplots for Categories

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x=df["Gender"])
plt.title("Gender Count")
plt.show()

sns.countplot(x=df["Age_Group"] if "Age_Group" in df.columns else df["Gender"])
plt.title("Category Frequency")
plt.show()

### Boxplots by Feature

In [ ]:
plt.figure(figsize=(15, 8))
for idx, col in enumerate(numerical_columns):
    plt.subplot(2, 3, idx + 1)
    sns.boxplot(y=df[col])
    plt.title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()

### Bivariate: Income versus Spending

In [ ]:
sns.scatterplot(
    x=df["Annual Income (k$)"],
    y=df["Spending Score (1-100)"],
)
plt.title("Income versus Spending Score")
plt.show()

Natural vertical bands appear in the cloud, an early hint that five income based groups are realistic.

### Pairplot

In [ ]:
sns.pairplot(df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]])
plt.show()

### Correlation Matrix and Heatmap

In [ ]:
correlation_matrix = df.corr()
correlation_matrix

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

## Feature Engineering

### Dropping the ID Column

CustomerID is just numbering with no pattern in it.

In [ ]:
if "CustomerID" in df.columns:
    df.drop("CustomerID", axis=1, inplace=True)
df.head()

### New Feature: Age Groups

Raw ages get pooled into Young, Adult, Middle Age and Senior so each segment is easier to talk about.

In [ ]:
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 25, 40, 60, 100],
    labels=["Young", "Adult", "Middle_Age", "Senior"],
)
df.head()

### New Feature: Income Categories

In [ ]:
df["Income_Category"] = pd.cut(
    df["Annual Income (k$)"],
    bins=[0, 40, 70, 150],
    labels=["Low", "Medium", "High"],
)
df.head()

### New Feature: Spending Categories

In [ ]:
df["Spending_Category"] = pd.cut(
    df["Spending Score (1-100)"],
    bins=[0, 40, 70, 100],
    labels=["Low", "Medium", "High"],
)
df.head()

### Encoding the Category Columns

In [ ]:
label_encoder = LabelEncoder()
for col in ["Age_Group", "Income_Category", "Spending_Category"]:
    df[col] = label_encoder.fit_transform(df[col])

df.head()

## Feature Scaling

K-Means measures distances, so a feature with a wide range would dominate the distance otherwise. Standardization brings every column to mean 0 and standard deviation 1.

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)
scaled_data.shape

In [ ]:
scaled_df = pd.DataFrame(scaled_data, columns=df.columns)
scaled_df.describe().round(3)

Means read at zero and standard deviations at one, which confirms the scaling worked.

## Finding the Optimal Number of Clusters

K must be chosen carefully. Too few merges distinct groups, too many splits one group into pieces. The two standard checks are the elbow on inertia and the silhouette score.

### Elbow Method

Inertia is the sum of squared distances from every point to its own centroid. It always falls as K grows, but after some K the drop slows sharply and that bend is the elbow.

In [ ]:
inertia_values = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, init="k-means++", random_state=42)
    kmeans.fit(scaled_data)
    inertia_values.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia_values, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

The curve bends sharply around K = 5 and then flattens, so five clusters is the candidate answer.

### Silhouette Score

The silhouette score measures how well each point sits inside its cluster versus the nearest rival cluster, from -1 to +1. Closer to 1 means tight, well separated clusters; around 0 means clusters overlap.

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, init="k-means++", random_state=42)
    cluster_labels = kmeans.fit_predict(scaled_data)
    score = silhouette_score(scaled_data, cluster_labels)
    silhouette_scores.append(score)
    print(f"K = {k}, Silhouette Score = {score:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score Analysis")
plt.show()

Both checks point in the same direction, so K = 5 is selected: the best balance between compactness and usefulness for the mall.

## Training the Final K-Means Model

The algorithm repeats four moves: place K centroids randomly, assign every point to the nearest centroid, recompute each centroid as the average of its members, and repeat until nothing changes. Inertia is the objective being minimized along the way.

In [ ]:
optimal_k = 5

kmeans = KMeans(n_clusters=optimal_k, init="k-means++", random_state=42)
cluster_labels = kmeans.fit_predict(scaled_data)

final_score = silhouette_score(scaled_data, cluster_labels)
print("Final silhouette score:", round(final_score, 4))

### Adding Cluster Labels and Checking the Sizes

In [ ]:
df["Cluster"] = cluster_labels
df.head()

In [ ]:
df["Cluster"].value_counts()

In [ ]:
sns.countplot(x=df["Cluster"])
plt.title("Customer Distribution per Cluster")
plt.show()

### Cluster Visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=df["Annual Income (k$)"],
    y=df["Spending Score (1-100)"],
    hue=df["Cluster"],
    palette="Set1",
    s=100,
)
plt.title("Customer Segmentation using K-Means")
plt.show()

### Cluster Centroids

Each centroid is the average customer of its cluster in scaled units.

In [ ]:
centroids = kmeans.cluster_centers_
centroids.round(3)

### Clusters with Centroids Overlaid

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=df["Annual Income (k$)"],
    y=df["Spending Score (1-100)"],
    hue=df["Cluster"],
    palette="Set1",
    s=100,
)
plt.scatter(
    centroids[:, 1],
    centroids[:, 2],
    s=300,
    c="black",
    marker="X",
    label="Centroids",
)
plt.title("Clusters with Centroids")
plt.legend()
plt.show()

## PCA Projection of the Clusters

The scaled data has several dimensions that cannot be drawn. PCA folds it into two components that keep most of the spread, which makes the separation visible.

In [ ]:
pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

pca_df = pd.DataFrame(pca_data, columns=["PCA1", "PCA2"])
pca_df["Cluster"] = cluster_labels
pca_df.head()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x="PCA1", y="PCA2", hue="Cluster", data=pca_df, palette="Set1", s=100)
plt.title("Clusters Projected onto Two Principal Components")
plt.show()

## Business Insights

Each cluster reads as a customer type:

| Cluster | Typical Profile | Marketing Angle |
|---------|-----------------|-----------------|
| High income, high spending | Premium customers | Loyalty programs, exclusive offers |
| High income, low spending | Careful big earners | Build trust, personalized products |
| Low income, high spending | Impulsive buyers | Limited-time deals |
| Low income, low spending | Minimal engagement | Low-cost promotions |
| Middle income, middle spending | The comfortable middle | General campaigns |

Uses for the business: targeted advertising, personalized offers, better retention, higher sales and sharper recommendations.

## Summary

The pipeline covered: cleaning, preprocessing, initial analysis, EDA, feature engineering with category columns, scaling, K-Means training, elbow and silhouette based selection of K, cluster visualization with centroids, and a PCA projection.

| Tool | Role |
|------|------|
| Python | Programming |
| Pandas | Data analysis |
| NumPy | Numerical operations |
| Matplotlib | Plotting |
| Seaborn | Plotting |
| Scikit-learn | Clustering, scaling, PCA |

The result: customers separated into five well-separated, visually confirmed segments, each with a clear business meaning.